# Kisan Mitra crop-photo classifier: lab photos vs field photos

The backend's photo check is a **linear probe**: a frozen, public DINOv2-small model turns a photo into
768 numbers, and a softmax regression trained here maps those to 38 crop and condition classes
(14 crops, from PlantVillage). The backend runs the same backbone as a public int8 ONNX file, so only
the small head (`backend/models/crop_head.json`) comes from this notebook, and refitting it on new
field photos takes minutes.

- **A**: head trained on PlantVillage only (leaves photographed on a plain background).
- **B**: PlantDoc's training photos (taken in fields and gardens) mixed in. This is what ships.

Both are scored on the PlantVillage test split and on PlantDoc's test photos, after removing test
photos that are near-duplicates of training photos. The "not sure" threshold is chosen on field
validation photos, never on the test photos. Cell 4 re-scores the test photos through the exact
CPU pipeline the backend runs.

Run with a GPU (Runtime > Change runtime type > T4 GPU). About 20 minutes.

In [ ]:
import os, subprocess, sys, time
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"  # public models only; never read a token
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "imagehash"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
T0 = time.time()

# Data: PlantVillage colour images and PlantDoc, both from their public GitHub repositories.
if not os.path.isdir("/content/pv/raw/color"):
    subprocess.run("git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset /content/pv", shell=True, check=True)
    subprocess.run("git -C /content/pv sparse-checkout set raw/color", shell=True, check=True)
if not os.path.isdir("/content/plantdoc/train"):
    subprocess.run("git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset /content/plantdoc", shell=True, check=True)
PV_ROOT = "/content/pv/raw/color"
PV_CLASSES = sorted(d for d in os.listdir(PV_ROOT) if os.path.isdir(os.path.join(PV_ROOT, d)))
CLS_IDX = {c: i for i, c in enumerate(PV_CLASSES)}

# PlantDoc folder -> PlantVillage class (lower-case match).
PD_TO_PV = {
    "apple scab leaf": "Apple___Apple_scab", "apple leaf": "Apple___healthy", "apple rust leaf": "Apple___Cedar_apple_rust",
    "bell_pepper leaf": "Pepper,_bell___healthy", "bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "blueberry leaf": "Blueberry___healthy", "cherry leaf": "Cherry_(including_sour)___healthy",
    "corn gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight", "corn rust leaf": "Corn_(maize)___Common_rust_",
    "peach leaf": "Peach___healthy", "potato leaf": "Potato___healthy", "potato leaf early blight": "Potato___Early_blight",
    "potato leaf late blight": "Potato___Late_blight", "raspberry leaf": "Raspberry___healthy",
    "soyabean leaf": "Soybean___healthy", "soybean leaf": "Soybean___healthy",
    "squash powdery mildew leaf": "Squash___Powdery_mildew", "strawberry leaf": "Strawberry___healthy",
    "tomato early blight leaf": "Tomato___Early_blight", "tomato septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "tomato leaf": "Tomato___healthy", "tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "tomato leaf late blight": "Tomato___Late_blight", "tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "tomato mold leaf": "Tomato___Leaf_Mold",
    "tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "grape leaf": "Grape___healthy", "grape leaf black rot": "Grape___Black_rot",
}
assert not set(PD_TO_PV.values()) - set(PV_CLASSES)

def pd_items(split):
    root = f"/content/plantdoc/{split}"; items, skipped = [], {}
    for d in sorted(os.listdir(root)):
        p = os.path.join(root, d)
        if not os.path.isdir(p): continue
        files = [os.path.join(p, f) for f in sorted(os.listdir(p)) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        t = PD_TO_PV.get(d.strip().lower())
        if t is None: skipped[d] = len(files); continue
        items += [(f, CLS_IDX[t]) for f in files]
    return items, skipped
pd_train_all, sk1 = pd_items("train"); pd_test_all, sk2 = pd_items("test")

from PIL import Image
from concurrent.futures import ThreadPoolExecutor
os.makedirs("/content/pd_small", exist_ok=True)
def shrink(item):
    src, y = item
    dst = os.path.splitext("/content/pd_small/" + src.replace("/content/plantdoc/", "").replace("/", "__"))[0] + ".jpg"
    if not os.path.exists(dst):
        try:
            im = Image.open(src).convert("RGB"); im.thumbnail((448, 448)); im.save(dst, quality=92)
        except Exception:
            return None
    return (dst, y, src)
with ThreadPoolExecutor(8) as ex:
    pd_train_all = [r for r in ex.map(shrink, pd_train_all) if r]
    pd_test_all = [r for r in ex.map(shrink, pd_test_all) if r]

# Drop test photos whose perceptual hash is within 6 bits of a training photo.
import imagehash, numpy as np
with ThreadPoolExecutor(8) as ex:
    H_tr = np.array(list(ex.map(lambda t: imagehash.phash(Image.open(t[0])).hash.flatten(), pd_train_all)))
    H_te = np.array(list(ex.map(lambda t: imagehash.phash(Image.open(t[0])).hash.flatten(), pd_test_all)))
dist = (H_te[:, None, :] != H_tr[None, :, :]).sum(-1).min(1)
pd_test = [t[:2] for t, d in zip(pd_test_all, dist) if d > 6]
pd_test_src = [t[2] for t, d in zip(pd_test_all, dist) if d > 6]
n_dupes = len(pd_test_all) - len(pd_test)

# Splits: PlantVillage stratified 80/10/10 by image; PlantDoc train 85/15 into train and validation.
import random
rng = random.Random(7)
by = {}
for c in PV_CLASSES:
    for f in sorted(os.listdir(os.path.join(PV_ROOT, c))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            by.setdefault(CLS_IDX[c], []).append((os.path.join(PV_ROOT, c, f), CLS_IDX[c]))
pv_train, pv_val, pv_test = [], [], []
for y, its in by.items():
    rng.shuffle(its); n = len(its); a, b = int(n * 0.8), int(n * 0.9)
    pv_train += its[:a]; pv_val += its[a:b]; pv_test += its[b:]
by = {}
for it in pd_train_all: by.setdefault(it[1], []).append(it[:2])
pd_train, pd_val = [], []
for y, its in by.items():
    rng.shuffle(its); k = max(1, int(round(len(its) * 0.15)))
    pd_val += its[:k]; pd_train += its[k:]
print("PlantVillage", len(pv_train), len(pv_val), len(pv_test), "| PlantDoc train", len(pd_train), "val", len(pd_val),
      "test", len(pd_test), "| near-duplicates removed", n_dupes, "| skipped", sk1, sk2)

In [ ]:
# Frozen DINOv2-small features: [CLS token, mean of patch tokens] from the final layer, 768 values per photo.
# The backend runs the same model as ONNX (onnx-community/dinov2-small), so the preprocessing below is the
# backend's, written once and used for both.
import torch, numpy as np, json, hashlib, base64
from transformers import AutoModel
from torch.utils.data import Dataset, DataLoader
DEV = "cuda"
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)

def preprocess(im):
    im = im.convert("RGB"); w, h = im.size
    nw, nh = (256, int(256 * h / w)) if w <= h else (int(256 * w / h), 256)
    im = im.resize((nw, nh), Image.BILINEAR)
    l, t = int(round((nw - 224) / 2.0)), int(round((nh - 224) / 2.0))
    x = (np.asarray(im.crop((l, t, l + 224, t + 224)), dtype=np.float32) / 255.0 - MEAN) / STD
    return x.transpose(2, 0, 1)

class DS(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return preprocess(Image.open(self.items[i][0]))

backbone = AutoModel.from_pretrained("facebook/dinov2-small").to(DEV).eval().half()
def pooled(h): return torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)

@torch.no_grad()
def embed(items):
    out = []
    for x in DataLoader(DS(items), batch_size=128, num_workers=os.cpu_count()):
        out.append(pooled(backbone(pixel_values=x.to(DEV).half()).last_hidden_state.float()).cpu())
    return torch.cat(out).numpy()

t = time.time()
F = {k: embed(v) for k, v in [("pv_train", pv_train), ("pv_val", pv_val), ("pv_test", pv_test),
                              ("pd_train", pd_train), ("pd_val", pd_val), ("pd_test", pd_test)]}
Y = {k: np.array([i[1] for i in v]) for k, v in [("pv_train", pv_train), ("pv_val", pv_val), ("pv_test", pv_test),
                                                ("pd_train", pd_train), ("pd_val", pd_val), ("pd_test", pd_test)]}
embed_seconds = round(time.time() - t)
print("features", {k: v.shape for k, v in F.items()}, embed_seconds, "s", flush=True)

In [ ]:
# Softmax regression on the features (full-batch L-BFGS on the GPU), with the standardisation folded into
# the weights so the backend needs only W and b.
from sklearn.metrics import f1_score
K = len(PV_CLASSES)

def fit(X, y, w, l2):
    mu, sd = X.mean(0), X.std(0) + 1e-6
    Xs = torch.tensor((X - mu) / sd, device=DEV); yt = torch.tensor(y, device=DEV); wt = torch.tensor(w / w.mean(), device=DEV, dtype=torch.float32)
    lin = torch.nn.Linear(X.shape[1], K).to(DEV)
    opt = torch.optim.LBFGS(lin.parameters(), lr=1, max_iter=500, history_size=20, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = (torch.nn.functional.cross_entropy(lin(Xs), yt, reduction="none") * wt).mean() + l2 * (lin.weight ** 2).sum()
        loss.backward(); return loss
    opt.step(closure)
    W = lin.weight.detach().cpu().numpy() / sd; b = lin.bias.detach().cpu().numpy() - W @ mu
    return W.astype(np.float32), b.astype(np.float32)

def proba(W, b, X):
    z = X @ W.T + b; z -= z.max(1, keepdims=True); p = np.exp(z); return p / p.sum(1, keepdims=True)

def crop(c): return c.split("___")[0]
def scores(P, y):
    yh = P.argmax(1); present = sorted(set(y.tolist()))
    return {"n": int(len(y)), "accuracy": round(float((yh == y).mean()), 4),
            "macro_f1": round(float(f1_score(y, yh, labels=present, average="macro", zero_division=0)), 4),
            "top3": round(float(np.mean([yy in np.argsort(-p)[:3] for p, yy in zip(P, y)])), 4),
            "crop_accuracy": round(float(np.mean([crop(PV_CLASSES[a]) == crop(PV_CLASSES[c]) for a, c in zip(yh, y)])), 4),
            "healthy_vs_diseased": round(float(np.mean([PV_CLASSES[a].endswith("healthy") == PV_CLASSES[c].endswith("healthy") for a, c in zip(yh, y)])), 4),
            "correct_of_n_per_class": {PV_CLASSES[c]: [int((yh[y == c] == c).sum()), int((y == c).sum())] for c in present}}

def curve(P, y):
    conf, ok = P.max(1), P.argmax(1) == y
    return [{"threshold": float(th), "coverage": round(float((conf >= th).mean()), 4),
             "accuracy_when_answered": round(float(ok[conf >= th].mean()) if (conf >= th).any() else 0.0, 4)}
            for th in np.round(np.arange(0.0, 0.96, 0.05), 2)]

def pick(cv, target=0.85):  # lowest threshold reaching the target accuracy on validation
    return next((r["threshold"] for r in cv if r["accuracy_when_answered"] >= target and r["coverage"] > 0), cv[-1]["threshold"])

def quantize(W):  # int8 per class row: what ships, so every reported test number uses it
    s = np.abs(W).max(1, keepdims=True) / 127.0
    return np.round(W / s).astype(np.int8), s.astype(np.float32)

RESULTS = {"backbone": "facebook/dinov2-small (frozen)", "features": "CLS + mean patch token, 768",
           "data": {k: int(len(v)) for k, v in Y.items()}, "pd_test_near_duplicates_removed": int(n_dupes),
           "embed_seconds": embed_seconds, "heads": {}}
L2S = [1e-5, 1e-4, 1e-3, 1e-2]
setups = {
    # A: what you get from PlantVillage alone, tuned on PlantVillage validation.
    "A_plantvillage_only": (F["pv_train"], Y["pv_train"], np.ones(len(Y["pv_train"])), "pv_val"),
    # B: field photos mixed in, weighted to about 30% of the loss, tuned on field validation photos.
    "B_plus_plantdoc": (np.concatenate([F["pv_train"], F["pd_train"]]), np.concatenate([Y["pv_train"], Y["pd_train"]]),
                        np.concatenate([np.full(len(Y["pv_train"]), 0.7 / len(Y["pv_train"])),
                                        np.full(len(Y["pd_train"]), 0.3 / len(Y["pd_train"]))]), "pd_val"),
}
heads = {}
for name, (X, y, w, val) in setups.items():
    best = None
    for l2 in L2S:
        W, b = fit(X, y, w, l2)
        acc = float((proba(W, b, F[val]).argmax(1) == Y[val]).mean())
        print(name, "l2", l2, val, "accuracy", round(acc, 4), flush=True)
        if best is None or acc > best[0]: best = (acc, l2, W, b)
    _, l2, W, b = best
    q, s = quantize(W); Wq = q.astype(np.float32) * s
    th = pick(curve(proba(Wq, b, F["pd_val"]), Y["pd_val"]))
    Ppd = proba(Wq, b, F["pd_test"])
    conf = Ppd.max(1); ok = Ppd.argmax(1) == Y["pd_test"]
    RESULTS["heads"][name] = {"l2": l2, "tuned_on": val, "plantvillage_test": scores(proba(Wq, b, F["pv_test"]), Y["pv_test"]),
        "plantdoc_test": scores(Ppd, Y["pd_test"]), "threshold_from_field_val": th,
        "plantdoc_test_at_threshold": {"coverage": round(float((conf >= th).mean()), 4),
                                       "accuracy_when_answered": round(float(ok[conf >= th].mean()), 4)},
        "field_val_curve": curve(proba(Wq, b, F["pd_val"]), Y["pd_val"])}
    heads[name] = (q, s, b, th)
    r = RESULTS["heads"][name]
    print(name, "PV test", r["plantvillage_test"]["accuracy"], "| PlantDoc test", r["plantdoc_test"]["accuracy"],
          "macro-F1", r["plantdoc_test"]["macro_f1"], "| threshold", th, r["plantdoc_test_at_threshold"], flush=True)

In [ ]:
# The shipped pipeline end to end: the public int8 ONNX backbone on CPU, plus head B. Scored on every
# PlantDoc test photo and 600 PlantVillage test photos, and timed per photo.
import onnxruntime as ort, urllib.request
REV = "8b1f705a3a7f6f062f6bdd21986c1583d3ef105d"
onnx_path = "/content/dinov2_small_int8.onnx"
if not os.path.exists(onnx_path):
    urllib.request.urlretrieve(f"https://huggingface.co/onnx-community/dinov2-small/resolve/{REV}/onnx/model_quantized.onnx", onnx_path)
sha_onnx = hashlib.sha256(open(onnx_path, "rb").read()).hexdigest()
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
inp, outs = sess.get_inputs()[0].name, [o.name for o in sess.get_outputs()]
def onnx_feat(path):
    h = sess.run(["last_hidden_state"], {inp: preprocess(Image.open(path))[None]})[0]
    return np.concatenate([h[:, 0], h[:, 1:].mean(1)], 1)[0]
q, s, b, th = heads["B_plus_plantdoc"]; Wq = q.astype(np.float32) * s
sub = random.Random(1).sample(range(len(pv_test)), 600)
lat = []
def run(items):
    X = []
    for p, _ in items:
        t = time.perf_counter(); X.append(onnx_feat(p)); lat.append((time.perf_counter() - t) * 1000)
    return np.stack(X)
Xo_pd = run(pd_test); Xo_pv = run([pv_test[i] for i in sub])
cos = float(np.mean(np.sum(Xo_pd * F["pd_test"], 1) / (np.linalg.norm(Xo_pd, axis=1) * np.linalg.norm(F["pd_test"], axis=1))))
Po = proba(Wq, b, Xo_pd); Pg = proba(Wq, b, F["pd_test"])
conf = Po.max(1); ok = Po.argmax(1) == Y["pd_test"]
RESULTS["shipped_onnx_cpu"] = {"onnx": "onnx-community/dinov2-small onnx/model_quantized.onnx", "revision": REV, "sha256": sha_onnx,
    "outputs": outs, "feature_cosine_vs_gpu_fp16": round(cos, 4), "argmax_agreement_vs_gpu": round(float((Po.argmax(1) == Pg.argmax(1)).mean()), 4),
    "plantdoc_test": scores(Po, Y["pd_test"]), "plantvillage_test_600": scores(proba(Wq, b, Xo_pv), Y["pv_test"][sub]),
    "plantdoc_test_at_threshold": {"threshold": th, "coverage": round(float((conf >= th).mean()), 4),
                                   "accuracy_when_answered": round(float(ok[conf >= th].mean()), 4)},
    "cpu_ms_per_photo_p50": round(float(np.percentile(lat, 50)), 1), "cpu_ms_per_photo_p95": round(float(np.percentile(lat, 95)), 1),
    "cpu_threads": os.cpu_count()}
# Three PlantDoc test photos to check the backend against, by their path in the PlantDoc repository.
RESULTS["check_photos"] = [{"plantdoc_path": pd_test_src[i].replace("/content/plantdoc/", ""),
                            "top": PV_CLASSES[int(Po[i].argmax())], "p": round(float(Po[i].max()), 4)} for i in (0, 50, 100)]
RESULTS["minutes_total"] = round((time.time() - T0) / 60, 1)
print(json.dumps(RESULTS["shipped_onnx_cpu"], indent=None)[:600])

In [ ]:
# Head B as base64 (int8 weights, float32 row scales and bias) in 4,000-character chunks, each with its own
# sha256, so it can be copied out of this page and checked piece by piece.
q, s, b, th = heads["B_plus_plantdoc"]
blob = base64.b64encode(q.tobytes() + s.astype(np.float32).tobytes() + b.astype(np.float32).tobytes()).decode()
HEAD = {"classes": PV_CLASSES, "dim": int(q.shape[1]), "threshold": th, "sha256": hashlib.sha256(blob.encode()).hexdigest()}
compact = {k: v for k, v in RESULTS.items()}
for k in compact["heads"]: compact["heads"][k].pop("field_val_curve", None)
print("===RESULTS===" + json.dumps(compact, separators=(",", ":")) + "===END===")
print("===HEAD===" + json.dumps(HEAD, separators=(",", ":")) + "===END===")
for i in range(0, len(blob), 4000):
    c = blob[i:i + 4000]
    print(f"===CHUNK {i // 4000} {hashlib.sha256(c.encode()).hexdigest()[:12]}===" + c + "===END===")
print("chunks", (len(blob) + 3999) // 4000, "length", len(blob))